# 04 — Rebuild reports from the validated partial matrix

**This is the adapted reporting notebook for the saved Notebook 03 run.** It does not compile MPI or rerun BFS. It reconstructs merged CSVs from checkpoint-validated configurations, excludes the six unfinished RMAT_s24 configurations, and rebuilds plots, the Markdown report, DOCX and PPTX.

The input matrix is **partial, not complete**: 234 of 240 planned configurations (97.5%) are expected. The generated report and presentation will disclose the missing configurations, the Colab CPU oversubscription, and the source-sampling limitation. No missing measurements are estimated or imputed.

Run all cells from top to bottom. Do not rerun Notebook 03.

In [1]:
from pathlib import Path
import os, sys, json, csv, math, shutil, traceback, hashlib
from datetime import datetime, timezone
from collections import defaultdict
from google.colab import drive

drive.mount('/content/drive')

WORKSPACE = Path('/content/drive/MyDrive/Distributed_BFS_MPI_Colab')
SOURCE_DIR = WORKSPACE / 'Distributed_BFS_MPI'
INPUT_RUN_DIR = WORKSPACE / 'runs' / 'full_matrix'
HELPER_PATH = WORKSPACE / 'colab_helpers.py'
WORK_DIR = Path('/content/Distributed_BFS_MPI')
EXPORT_DIR = WORKSPACE / 'exports' / 'reports_full_matrix_partial'
ARCHIVE_NAME = '04_full_matrix_partial_reports_results.zip'

if not SOURCE_DIR.is_dir():
    raise FileNotFoundError(f'Missing project source directory: {SOURCE_DIR}')
if not HELPER_PATH.is_file():
    raise FileNotFoundError(f'Missing shared helper module: {HELPER_PATH}')
if not INPUT_RUN_DIR.is_dir():
    raise FileNotFoundError(f'Missing saved full-matrix run: {INPUT_RUN_DIR}')

sys.path.insert(0, str(WORKSPACE))
import colab_helpers as cb
cb.clear_previous_archive(WORKSPACE, ARCHIVE_NAME)

print('Input run:', INPUT_RUN_DIR)
print('Report export:', EXPORT_DIR)
print('Archive on success:', ARCHIVE_NAME)

Mounted at /content/drive
Input run: /content/drive/MyDrive/Distributed_BFS_MPI_Colab/runs/full_matrix
Report export: /content/drive/MyDrive/Distributed_BFS_MPI_Colab/exports/reports_full_matrix_partial
Archive on success: 04_full_matrix_partial_reports_results.zip


## 1. Audit saved checkpoints and reconstruct the measured CSVs

This validation deliberately does **not** call `require_matrix_complete()`, because that helper correctly rejects incomplete matrices. Instead, each present `.done` checkpoint is revalidated with the same per-configuration validator used by Notebook 03. Only those checkpointed folders are merged.

In [2]:
# Expected matrix from the frozen Notebook 03 manifest.
run_config_path = INPUT_RUN_DIR / 'run_config.json'
if not run_config_path.is_file():
    raise FileNotFoundError(f'Missing run_config.json: {run_config_path}')
run_config = json.loads(run_config_path.read_text(encoding='utf-8'))

EXPECTED_DATASETS = list(run_config['datasets'])
EXPECTED_RANKS = [int(x) for x in run_config['ranks']]
EXPECTED_ORDERINGS = list(run_config['orderings'])
EXPECTED_MODES = list(run_config['modes'])
REPETITIONS = int(run_config['repetitions'])
EXPECTED_TAGS = {
    f'{ds}_p{p}_{ordering}_{mode}'
    for ds in EXPECTED_DATASETS for p in EXPECTED_RANKS
    for ordering in EXPECTED_ORDERINGS for mode in EXPECTED_MODES
}

EXPECTED_MISSING = {
    'RMAT_s24_p8_random_blocking',
    'RMAT_s24_p8_random_nonblocking',
    'RMAT_s24_p16_original_blocking',
    'RMAT_s24_p16_original_nonblocking',
    'RMAT_s24_p16_random_blocking',
    'RMAT_s24_p16_random_nonblocking',
}

if len(EXPECTED_TAGS) != 240 or len(EXPECTED_DATASETS) != 12 or REPETITIONS != 16:
    raise RuntimeError(
        f'Unexpected frozen experiment matrix: {len(EXPECTED_TAGS)} configurations, '
        f'{len(EXPECTED_DATASETS)} datasets, {REPETITIONS} sources/configuration.'
    )
if run_config.get('runner_version') != cb.RUNNER_VERSION:
    raise RuntimeError('Notebook 03 runner version does not match the installed validation helper.')
source_hash = cb.source_fingerprint(SOURCE_DIR)
if run_config.get('source_fingerprint') != source_hash:
    raise RuntimeError('Project source fingerprint differs from the source used by Notebook 03.')

checkpoint_dir = INPUT_RUN_DIR / 'checkpoints'
config_dir = INPUT_RUN_DIR / 'configs'
log_dir = INPUT_RUN_DIR / 'logs'
done_tags = {p.stem for p in checkpoint_dir.glob('*.done')}
config_dirs = {p.name for p in config_dir.iterdir() if p.is_dir()} if config_dir.exists() else set()
log_tags = {p.stem for p in log_dir.glob('*.log')} if log_dir.exists() else set()
missing_tags = EXPECTED_TAGS - done_tags
extra_tags = done_tags - EXPECTED_TAGS

if extra_tags:
    raise RuntimeError(f'Unexpected checkpoint tags: {sorted(extra_tags)}')
if missing_tags != EXPECTED_MISSING:
    raise RuntimeError(
        'The incomplete configuration set differs from the audited archive. '
        f'Expected missing={sorted(EXPECTED_MISSING)}, observed missing={sorted(missing_tags)}'
    )
if len(done_tags) != 234:
    raise RuntimeError(f'Expected 234 validated checkpoints, found {len(done_tags)}')
if not config_dirs.issubset(EXPECTED_TAGS) or not log_tags.issubset(EXPECTED_TAGS):
    raise RuntimeError('Unexpected configuration directory or run-log tag found.')

# Confirm that this is the known resource-killed Notebook 03 attempt.
failure_path = INPUT_RUN_DIR / 'FAILURE.json'
if not failure_path.is_file():
    raise RuntimeError('The partial full-matrix run has no FAILURE.json provenance record.')
failure_info = json.loads(failure_path.read_text(encoding='utf-8'))
if failure_info.get('notebook') != '03' or '137' not in str(failure_info.get('error', '')):
    raise RuntimeError('The saved full-matrix failure record does not match the audited exit-137 interruption.')

# Verify recorded Open MPI probes and SNAP cache files.
toolchain_path = INPUT_RUN_DIR / 'metadata' / 'openmpi_toolchain.json'
if not toolchain_path.is_file():
    raise FileNotFoundError(f'Missing Open MPI metadata: {toolchain_path}')
toolchain = json.loads(toolchain_path.read_text(encoding='utf-8'))
probe_results = toolchain.get('probes', {})
for p in EXPECTED_RANKS:
    probe = probe_results.get(str(p), {})
    if probe.get('passed') is not True:
        raise RuntimeError(f'Open MPI process-count probe {p} did not pass in saved metadata.')

snap_manifest_path = INPUT_RUN_DIR / 'snap_cache_manifest.json'
if not snap_manifest_path.is_file():
    raise FileNotFoundError(f'Missing SNAP cache manifest: {snap_manifest_path}')
snap_manifest = json.loads(snap_manifest_path.read_text(encoding='utf-8'))
for filename, details in snap_manifest.items():
    snap_file = WORKSPACE / 'data' / 'snap' / filename
    if not snap_file.is_file():
        raise FileNotFoundError(f'Missing cached SNAP graph: {snap_file}')
    digest = hashlib.sha256(snap_file.read_bytes()).hexdigest()
    if snap_file.stat().st_size != int(details['bytes']) or digest != details['sha256']:
        raise RuntimeError(f'SNAP cache integrity check failed for {filename}')
    if run_config.get('snap_sha256', {}).get(filename) != digest:
        raise RuntimeError(f'SNAP hash differs from frozen run_config for {filename}')

# Revalidate every completed configuration and merge its CSVs into the separate export folder.
if EXPORT_DIR.exists():
    shutil.rmtree(EXPORT_DIR)
for child in ('csv', 'plots', 'report', 'presentation', 'metadata'):
    (EXPORT_DIR / child).mkdir(parents=True, exist_ok=True)

CSV_NAMES = ('summary.csv', 'per_level.csv', 'rank_metrics.csv')
row_counts = {name: 0 for name in CSV_NAMES}
source_outcomes = {}
graph_metadata = {}
rank_outcomes = {}

for tag in sorted(done_tags):
    cfg_path = config_dir / tag
    # Existing project validator checks 16 source rows, PASS status, all rank IDs,
    # per-level source coverage, identity fields, and the run.log completion line.
    summary_check = cb.read_csv(cfg_path / 'summary.csv')
    first = summary_check[0]
    ds = str(first['dataset'])
    ranks_expected = int(first['ranks'])
    ordering = str(first['ordering'])
    mode = str(first['mode'])
    cb.validate_single_config(
        cfg_path, ranks_expected, REPETITIONS,
        expected_dataset=ds, expected_ordering=ordering, expected_mode=mode,
    )

    if tag != f'{ds}_p{ranks_expected}_{ordering}_{mode}':
        raise RuntimeError(f'Folder/configuration identity mismatch: {tag}')

    # Stronger cross-configuration data checks in addition to the project validator.
    with (cfg_path / 'summary.csv').open(newline='', encoding='utf-8') as handle:
        rows = list(csv.DictReader(handle))
    if len(rows) != REPETITIONS or sorted(int(x['source_index']) for x in rows) != list(range(REPETITIONS)):
        raise RuntimeError(f'{tag}: summary source rows are incomplete or duplicated.')
    for row in rows:
        if row['validation'].upper() != 'PASS':
            raise RuntimeError(f'{tag}: a BFS source validation did not pass.')
        numeric = ['vertices','edges','ranks','source_index','source_vertex','elapsed_ms',
                   'traversed_edges','teps','reachable_vertices','remote_pairs_sent',
                   'remote_payload_bytes','graph_seed','permutation_seed']
        vals = {key: float(row[key]) for key in numeric}
        if not all(math.isfinite(value) for value in vals.values()):
            raise RuntimeError(f'{tag}: non-finite summary metric.')
        if any(value < 0 for key, value in vals.items() if key != 'source_vertex'):
            raise RuntimeError(f'{tag}: negative summary metric.')
        elapsed_ms = vals['elapsed_ms']
        expected_teps = vals['traversed_edges'] / (elapsed_ms / 1000.0) if elapsed_ms > 0 else None
        if expected_teps is None or not math.isclose(vals['teps'], expected_teps, rel_tol=1e-4, abs_tol=1.0):
            raise RuntimeError(f'{tag}: TEPS does not agree with traversed_edges / elapsed_ms.')
        graph_key = ds
        graph_value = (int(vals['vertices']), int(vals['edges']))
        if graph_key in graph_metadata and graph_metadata[graph_key] != graph_value:
            raise RuntimeError(f'{tag}: graph vertex/edge totals differ across configurations.')
        graph_metadata[graph_key] = graph_value
        source_key = (ds, int(row['source_index']))
        source_value = (int(vals['traversed_edges']), int(vals['reachable_vertices']))
        if source_key in source_outcomes and source_outcomes[source_key] != source_value:
            raise RuntimeError(f'{tag}: sampled-source BFS outcome differs across rank/order/mode variants.')
        source_outcomes[source_key] = source_value

    with (cfg_path / 'per_level.csv').open(newline='', encoding='utf-8') as handle:
        level_rows = list(csv.DictReader(handle))
    level_sources = set()
    for row in level_rows:
        if row['dataset'] != ds or row['ordering'] != ordering or row['mode'] != mode or int(row['ranks']) != ranks_expected:
            raise RuntimeError(f'{tag}: per-level identity mismatch.')
        level_sources.add(int(row['source_index']))
        for key in ['vertices','ranks','source_index','level','next_frontier','expansion_ms',
                    'packing_ms','count_exchange_ms','collective_request_ms','local_candidate_ms',
                    'residual_wait_ms','received_merge_ms','level_wall_ms','remote_pairs_sent',
                    'remote_payload_bytes','frontier_imbalance']:
            value = float(row[key])
            if not math.isfinite(value) or value < 0:
                raise RuntimeError(f'{tag}: invalid per-level metric {key}={row[key]}')
    if level_sources != set(range(REPETITIONS)):
        raise RuntimeError(f'{tag}: per-level measurements do not cover all 16 sources.')

    with (cfg_path / 'rank_metrics.csv').open(newline='', encoding='utf-8') as handle:
        rank_rows = list(csv.DictReader(handle))
    for row in rank_rows:
        if row['dataset'] != ds or row['ordering'] != ordering or int(row['ranks']) != ranks_expected:
            raise RuntimeError(f'{tag}: rank-metric identity mismatch.')
        key = (ds, ordering, ranks_expected, int(row['rank']))
        value = (int(row['local_adjacency_entries']), int(row['local_cut_edges']))
        if key in rank_outcomes and rank_outcomes[key] != value:
            raise RuntimeError(f'{tag}: static rank metrics differ between blocking/non-blocking variants.')
        rank_outcomes[key] = value

# Confirm all 16 sampled source IDs exist for every dataset and all expected datasets are represented.
for ds in EXPECTED_DATASETS:
    source_ids = {source_id for dataset, source_id in source_outcomes if dataset == ds}
    if source_ids != set(range(REPETITIONS)):
        raise RuntimeError(f'{ds}: expected source indices 0..15, found {sorted(source_ids)}')
if set(graph_metadata) != set(EXPECTED_DATASETS):
    raise RuntimeError('The validated checkpoints do not cover all 12 configured datasets.')

# Merge only the 234 validated folders. The uncheckpointed RMAT_s24_p8_random_blocking
# folder is intentionally ignored even if it contains partial CSV files.
for name in CSV_NAMES:
    target = EXPORT_DIR / 'csv' / name
    header = None
    with target.open('w', newline='', encoding='utf-8') as out_handle:
        writer = None
        for tag in sorted(done_tags):
            source_path = config_dir / tag / name
            if not source_path.is_file() or source_path.stat().st_size == 0:
                raise RuntimeError(f'Validated configuration unexpectedly lacks {name}: {tag}')
            with source_path.open(newline='', encoding='utf-8') as in_handle:
                reader = csv.reader(in_handle)
                current_header = next(reader, None)
                if not current_header:
                    raise RuntimeError(f'Empty CSV header: {source_path}')
                if header is None:
                    header = current_header
                    writer = csv.writer(out_handle)
                    writer.writerow(header)
                elif current_header != header:
                    raise RuntimeError(f'CSV schema mismatch in {source_path}')
                for data_row in reader:
                    writer.writerow(data_row)
                    row_counts[name] += 1

expected_summary_rows = len(done_tags) * REPETITIONS
if row_counts['summary.csv'] != expected_summary_rows or expected_summary_rows != 3744:
    raise RuntimeError(f'Expected 3,744 validated summary rows, found {row_counts["summary.csv"]}.')
if row_counts['per_level.csv'] == 0 or row_counts['rank_metrics.csv'] == 0:
    raise RuntimeError('Merged per-level or rank metrics CSV is empty.')

# Prepare source-level sampling diagnostics using one representative P=1/original/blocking
# configuration per dataset; cross-configuration consistency was verified above.
import pandas as pd
summary_df = pd.read_csv(EXPORT_DIR / 'csv' / 'summary.csv')
representative = summary_df[
    (summary_df['ranks'] == 1) &
    (summary_df['ordering'] == 'original') &
    (summary_df['mode'] == 'blocking')
].copy()
sampling_rows = []
for ds in sorted(EXPECTED_DATASETS):
    g = representative[representative['dataset'] == ds]
    if g['source_index'].nunique() != REPETITIONS:
        raise RuntimeError(f'{ds}: representative source sample does not contain 16 distinct sources.')
    zero_count = int((g['traversed_edges'] == 0).sum())
    sampling_rows.append({
        'dataset': ds,
        'sampled_sources': int(g['source_index'].nunique()),
        'zero_work_sources': zero_count,
        'active_sources': int((g['traversed_edges'] > 0).sum()),
        'zero_work_share': zero_count / REPETITIONS,
    })
sampling_df = pd.DataFrame(sampling_rows)
sampling_df.to_csv(EXPORT_DIR / 'csv' / 'source_sampling_audit.csv', index=False)

coverage_rows = []
for ds in sorted(EXPECTED_DATASETS):
    tags_for_ds = [tag for tag in done_tags if tag.startswith(ds + '_p')]
    rank_sets = []
    for ordering in EXPECTED_ORDERINGS:
        for mode in EXPECTED_MODES:
            rank_sets.append({
                int(tag.split('_p', 1)[1].split('_', 1)[0])
                for tag in tags_for_ds
                if tag.endswith(f'_{ordering}_{mode}')
            })
    common_ranks = set.intersection(*rank_sets) if rank_sets else set()
    if not common_ranks:
        raise RuntimeError(f'{ds}: no common rank count for all ordering/mode comparisons.')
    completed = len(tags_for_ds)
    coverage_rows.append({
        'dataset': ds,
        'planned_configurations': len(EXPECTED_RANKS) * len(EXPECTED_ORDERINGS) * len(EXPECTED_MODES),
        'validated_configurations': completed,
        'missing_configurations': len(EXPECTED_RANKS) * len(EXPECTED_ORDERINGS) * len(EXPECTED_MODES) - completed,
        'highest_common_rank_for_per_level_plot': max(common_ranks),
    })
coverage_df = pd.DataFrame(coverage_rows)
coverage_df.to_csv(EXPORT_DIR / 'csv' / 'configuration_coverage.csv', index=False)

zero_work_runs = int((summary_df['traversed_edges'] == 0).sum())
active_work_runs = int((summary_df['traversed_edges'] > 0).sum())
logical_cpus = toolchain.get('cpu_count')
MATRIX_INFO = {
    'datasets': sorted(EXPECTED_DATASETS),
    'ranks': sorted(EXPECTED_RANKS),
    'orderings': EXPECTED_ORDERINGS,
    'modes': EXPECTED_MODES,
    'planned_configurations': len(EXPECTED_TAGS),
    'configurations': len(done_tags),
    'completion_fraction': len(done_tags) / len(EXPECTED_TAGS),
    'bfs_summary_rows': row_counts['summary.csv'],
    'per_level_rows': row_counts['per_level.csv'],
    'rank_metric_rows': row_counts['rank_metrics.csv'],
    'repetitions_per_configuration': REPETITIONS,
    'missing_configurations': sorted(missing_tags),
    'zero_work_runs': zero_work_runs,
    'active_work_runs': active_work_runs,
    'zero_work_fraction': zero_work_runs / len(summary_df),
    'sampled_source_count': int(sampling_df['sampled_sources'].sum()),
    'zero_work_unique_sampled_sources': int(sampling_df['zero_work_sources'].sum()),
    'logical_cpus': logical_cpus,
}

# Store explicit partial-run provenance; deliberately do not create RUN_COMPLETE.json.
(EXPORT_DIR / 'metadata' / 'input_run_config.json').write_text(
    json.dumps(run_config, indent=2), encoding='utf-8'
)
shutil.copy2(failure_path, EXPORT_DIR / 'metadata' / 'input_FAILURE.json')
shutil.copy2(toolchain_path, EXPORT_DIR / 'metadata' / 'input_openmpi_toolchain.json')
shutil.copy2(snap_manifest_path, EXPORT_DIR / 'metadata' / 'input_snap_cache_manifest.json')
environment_note = (
    f"MPI implementation: {toolchain.get('mpi_implementation', 'unknown')}\n"
    f"Compiler: {toolchain.get('compiler', 'unknown')}\n"
    f"Compiler version: {toolchain.get('compiler_version', 'unknown')}\n"
    f"Launcher: {toolchain.get('launcher', 'unknown')}\n"
    f"Launcher version: {toolchain.get('launcher_version', 'unknown')}\n"
    f"Available logical CPUs recorded by Notebook 03: {logical_cpus}\n"
    "Platform: single Google Colab VM; MPI ranks are not distributed across multiple nodes.\n"
    "Interpretation: rank counts greater than the available logical CPU count are oversubscribed.\n"
)
(EXPORT_DIR / 'metadata' / 'input_environment_summary.txt').write_text(environment_note, encoding='utf-8')
(EXPORT_DIR / 'metadata' / 'partial_matrix_validation.json').write_text(
    json.dumps({
        'input_run': 'full_matrix',
        'matrix_status': 'PARTIAL_VALIDATED',
        'all_included_configurations_validated': True,
        'benchmark_matrix_complete': False,
        **MATRIX_INFO,
        'source_fingerprint': source_hash,
        'runner_version': run_config.get('runner_version'),
        'failure_exit_code': 137,
        'failed_configuration_at_interruption': 'RMAT_s24_p8_random_blocking',
    }, indent=2), encoding='utf-8'
)

# Copy project scripts to a temporary work area. The canonical Drive source is not modified.
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
shutil.copytree(SOURCE_DIR, WORK_DIR)

print('PARTIAL MATRIX AUDIT PASSED')
print(f"Validated configurations: {MATRIX_INFO['configurations']}/{MATRIX_INFO['planned_configurations']} ({MATRIX_INFO['completion_fraction']:.1%})")
print('Validated BFS source runs:', MATRIX_INFO['bfs_summary_rows'])
print('Merged CSV rows:', row_counts)
print('Zero-work BFS runs:', f"{zero_work_runs:,}/{len(summary_df):,} ({MATRIX_INFO['zero_work_fraction']:.1%})")
print('Distinct sampled source IDs with zero traversed edges:', MATRIX_INFO['zero_work_unique_sampled_sources'], '/', MATRIX_INFO['sampled_source_count'])
print('Missing configurations:', *MATRIX_INFO['missing_configurations'], sep='\n  - ')
print('Recorded logical CPUs:', logical_cpus)
print('Export staging directory:', EXPORT_DIR)

PARTIAL MATRIX AUDIT PASSED
Validated configurations: 234/240 (97.5%)
Validated BFS source runs: 3744
Merged CSV rows: {'summary.csv': 3744, 'per_level.csv': 211144, 'rank_metrics.csv': 1408}
Zero-work BFS runs: 2,102/3,744 (56.1%)
Distinct sampled source IDs with zero traversed edges: 109 / 192
Missing configurations:
  - RMAT_s24_p16_original_blocking
  - RMAT_s24_p16_original_nonblocking
  - RMAT_s24_p16_random_blocking
  - RMAT_s24_p16_random_nonblocking
  - RMAT_s24_p8_random_blocking
  - RMAT_s24_p8_random_nonblocking
Recorded logical CPUs: 2
Export staging directory: /content/drive/MyDrive/Distributed_BFS_MPI_Colab/exports/reports_full_matrix_partial


## 2. Rebuild charts and documents with partial-matrix disclosures

The original analysis scripts assume a complete matrix. This cell patches only the temporary `/content` copy of the plotting/report scripts. The original C++ source, saved run folders and checkpoint data remain unchanged.

Changes in this report build:
- Add zero-work/active-source counts and active-only medians to `summary_aggregated.csv` while retaining all original source rows and all-source medians.
- Keep per-level comparisons at the highest rank count shared by all four ordering/communication combinations for a dataset.
- Use a consistent colour normalization across both orderings in the edge-load imbalance plot.
- Prefer the fully sampled `roadNet-CA` charts in the presentation and add a slide disclosing coverage and runtime limits.

In [3]:
def replace_once(path: Path, old: str, new: str, label: str):
    text = path.read_text(encoding='utf-8')
    count = text.count(old)
    if count != 1:
        raise RuntimeError(f'Expected exactly one patch target for {label}; found {count}.')
    path.write_text(text.replace(old, new, 1), encoding='utf-8')

analysis_script = WORK_DIR / 'scripts' / 'analyze_results.py'
report_script = WORK_DIR / 'scripts' / 'generate_report.py'
docx_script = WORK_DIR / 'scripts' / 'generate_report_docx.py'
presentation_script = WORK_DIR / 'scripts' / 'make_presentation.py'

old_agg = """agg = summary.groupby(\n    ['dataset', 'ordering', 'mode', 'ranks'], as_index=False\n).agg(\n    median_teps=('teps', 'median'),\n    median_elapsed_ms=('elapsed_ms', 'median'),\n    median_remote_pairs=('remote_pairs_sent', 'median'),\n    median_reachable=('reachable_vertices', 'median'),\n    sources=('source_index', 'nunique'),\n)\nagg['speedup'] = float('nan')\nagg['efficiency'] = float('nan')\nfor (_, _, _), group in agg.groupby(['dataset', 'ordering', 'mode']):\n    baseline = group.loc[group.ranks == 1, 'median_elapsed_ms']\n    if not baseline.empty and float(baseline.iloc[0]) > 0:\n        base = float(baseline.iloc[0])\n        agg.loc[group.index, 'speedup'] = base / agg.loc[group.index, 'median_elapsed_ms']\n        agg.loc[group.index, 'efficiency'] = (\n            agg.loc[group.index, 'speedup'] / agg.loc[group.index, 'ranks'] * 100\n        )\nagg.to_csv(out / 'summary_aggregated.csv', index=False)\n"""
new_agg = """group_keys = ['dataset', 'ordering', 'mode', 'ranks']\nagg = summary.groupby(group_keys, as_index=False).agg(\n    median_teps=('teps', 'median'),\n    median_elapsed_ms=('elapsed_ms', 'median'),\n    median_remote_pairs=('remote_pairs_sent', 'median'),\n    median_reachable=('reachable_vertices', 'median'),\n    sources=('source_index', 'nunique'),\n    zero_work_sources=('traversed_edges', lambda x: int((x == 0).sum())),\n)\nactive = summary[summary.traversed_edges > 0].groupby(group_keys, as_index=False).agg(\n    median_active_teps=('teps', 'median'),\n    median_active_elapsed_ms=('elapsed_ms', 'median'),\n    nonzero_sources=('source_index', 'nunique'),\n)\nagg = agg.merge(active, on=group_keys, how='left', validate='one_to_one')\nagg['nonzero_sources'] = agg['nonzero_sources'].fillna(0).astype(int)\nagg['zero_work_fraction'] = agg['zero_work_sources'] / agg['sources']\nagg['active_elapsed_ratio_p1_over_p'] = float('nan')\nagg['active_elapsed_ratio_percent_per_rank'] = float('nan')\nfor (_, _, _), group in agg.groupby(['dataset', 'ordering', 'mode']):\n    baseline = group.loc[group.ranks == 1, 'median_active_elapsed_ms']\n    if not baseline.empty and pd.notna(baseline.iloc[0]) and float(baseline.iloc[0]) > 0:\n        base = float(baseline.iloc[0])\n        valid = group['median_active_elapsed_ms'].notna() & (group['median_active_elapsed_ms'] > 0)\n        idx = group.index[valid]\n        agg.loc[idx, 'active_elapsed_ratio_p1_over_p'] = base / agg.loc[idx, 'median_active_elapsed_ms']\n        agg.loc[idx, 'active_elapsed_ratio_percent_per_rank'] = (\n            agg.loc[idx, 'active_elapsed_ratio_p1_over_p'] / agg.loc[idx, 'ranks'] * 100\n        )\nagg.to_csv(out / 'summary_aggregated.csv', index=False)\n"""
replace_once(analysis_script, old_agg, new_agg, 'source-aware aggregation')
replace_once(
    analysis_script,
    "    ax.set(title=f'TEPS scaling — {dataset}', xlabel='MPI ranks', ylabel='Median TEPS')\n",
    "    ax.set(title=f'Median TEPS across all sampled sources (includes zero-edge runs) — {dataset}', xlabel='MPI ranks', ylabel='Median TEPS')\n",
    'all-source TEPS chart label',
)
replace_once(
    analysis_script,
    """for dataset, group in levels.groupby('dataset'):\n    max_ranks = int(group.ranks.max())\n    group = group[group.ranks == max_ranks]\n""",
    """for dataset, group in levels.groupby('dataset'):\n    available = agg[agg.dataset == dataset]\n    rank_sets = []\n    for ordering in sorted(summary.ordering.unique()):\n        for mode in sorted(summary['mode'].unique()):\n            rank_sets.append(set(available.loc[(available.ordering == ordering) & (available['mode'] == mode), 'ranks'].astype(int)))\n    common_ranks = set.intersection(*rank_sets) if rank_sets else set()\n    if not common_ranks:\n        raise SystemExit(f'No common rank count across ordering/mode combinations for per-level plot: {dataset}')\n    max_ranks = max(common_ranks)\n    group = group[group.ranks == max_ranks]\n""",
    'common-rank per-level comparisons',
)
old_scatter = """    markers = {'original': 'o', 'random': '^'}\n    scatter = None\n    for ordering, config in compare.groupby('ordering'):\n        scatter = ax.scatter(\n            config.cut_fraction, config.median_teps,\n            c=config.edge_load_imbalance, s=55 + config.ranks * 8,\n            marker=markers.get(ordering, 's'), label=ordering,\n        )\n"""
new_scatter = """    markers = {'original': 'o', 'random': '^'}\n    scatter = None\n    color_values = compare.edge_load_imbalance.dropna()\n    color_min = float(color_values.min()) if not color_values.empty else 0.0\n    color_max = float(color_values.max()) if not color_values.empty else 1.0\n    if color_max <= color_min:\n        color_max = color_min + 1.0\n    for ordering, config in compare.groupby('ordering'):\n        scatter = ax.scatter(\n            config.cut_fraction, config.median_teps,\n            c=config.edge_load_imbalance, vmin=color_min, vmax=color_max,\n            s=55 + config.ranks * 8, marker=markers.get(ordering, 's'), label=ordering,\n        )\n"""
replace_once(analysis_script, old_scatter, new_scatter, 'shared edge-imbalance colour scale')

# Presentation charts should use a dataset with 16/16 sampled sources having nonzero work.
pres_text = presentation_script.read_text(encoding='utf-8')
def replace_text_once(text, old, new, label):
    count = text.count(old)
    if count != 1:
        raise RuntimeError(f'Expected exactly one patch target for {label}; found {count}.')
    return text.replace(old, new, 1)

old_select = " selected_plots[label] = matches[0]"
new_select = " preferred = [f for f in matches if f.stem.endswith('_roadNet-CA')]\n selected_plots[label] = preferred[0] if preferred else matches[0]"
pres_text = replace_text_once(pres_text, old_select, new_select, 'presentation plot selection')
pres_text = replace_text_once(
    pres_text,
    "'Strong-scaling speedup and parallel efficiency relative to P=1.'",
    "'Active-source elapsed-time ratios are exploratory under Colab CPU oversubscription.'",
    'presentation scaling metric wording',
)
coverage_slide = """slide('Coverage and interpretation limits',[\n '234 of 240 planned configurations validated (97.5%); 3,744 source-level BFS runs are included. Six RMAT_s24 cases are incomplete and excluded; no values are estimated or imputed.',\n 'The recorded Colab runtime had 2 logical CPUs. P=4, 8 and 16 runs were oversubscribed on one VM; these are not dedicated-core or multi-node scaling results.',\n '2,102 of 3,744 recorded BFS runs (56.1%) traversed zero edges. The 16 fixed-seed sampled sources per dataset were not degree-stratified.',\n 'For RMAT_s17 through RMAT_s24, all-source median TEPS is zero. Active-source aggregates are supplied separately with their sample counts; treat these small samples as exploratory.',\n 'Per-level comparisons use the highest rank count shared by all four ordering/mode combinations; for RMAT_s24 this is P=4.'\n])\n"""
conclusion_line = "slide('Conclusion and limitations',['The implementation tests how graph structure and vertex ordering affect communication.','MPI_Ialltoallv may or may not improve end-to-end time, depending on MPI progress and available local work.','Large-graph serial-reference validation is skipped above the configured threshold; structural validation remains active.','This is an educational 1D distributed BFS implementation, not full Graph500 compliance.'])\n"
pres_text = replace_text_once(pres_text, conclusion_line, conclusion_line + coverage_slide, 'partial-matrix presentation slide')
presentation_script.write_text(pres_text, encoding='utf-8')

# Install only the report-generation dependencies; MPI is never invoked in Notebook 04.
cb.run_stream([
    sys.executable, '-m', 'pip', 'install', '-q', '-r', str(WORK_DIR / 'requirements.txt')
], timeout=900, check=True)

cb.run_stream([
    sys.executable, str(analysis_script),
    '--input-dir', str(EXPORT_DIR / 'csv'),
    '--output-dir', str(EXPORT_DIR / 'plots'),
], cwd=WORK_DIR, timeout=900, check=True)

# Audit the derived aggregate schema and coverage before any documents are generated.
aggregated_path = EXPORT_DIR / 'plots' / 'summary_aggregated.csv'
aggregated_df = pd.read_csv(aggregated_path)
if len(aggregated_df) != 234 or set(aggregated_df['sources'].astype(int)) != {16}:
    raise RuntimeError('summary_aggregated.csv does not contain 234 configurations with 16 sources each.')
if not {'zero_work_sources','nonzero_sources','median_active_teps','active_elapsed_ratio_p1_over_p'}.issubset(aggregated_df.columns):
    raise RuntimeError('Source-aware aggregate columns are missing.')
if not ((aggregated_df['zero_work_sources'] + aggregated_df['nonzero_sources']) == aggregated_df['sources']).all():
    raise RuntimeError('Zero-work and active-source counts do not sum to the source count.')

# Markdown report uses the original measured medians, then adds explicit coverage/sampling disclosure.
environment_file = EXPORT_DIR / 'metadata' / 'input_environment_summary.txt'
md_path = EXPORT_DIR / 'report' / 'Measured_Analysis_Report.md'
cb.run_stream([
    sys.executable, str(report_script),
    '--input-dir', str(EXPORT_DIR / 'csv'),
    '--output', str(md_path),
    '--environment', str(environment_file),
], cwd=WORK_DIR, timeout=900, check=True)

missing_text = '; '.join(f'`{x}`' for x in MATRIX_INFO['missing_configurations'])
coverage_md = [
    '', '', '## Partial-matrix coverage and interpretation limits', '',
    f"- **Coverage:** {MATRIX_INFO['configurations']}/{MATRIX_INFO['planned_configurations']} planned configurations validated ({MATRIX_INFO['completion_fraction']:.1%}), representing {MATRIX_INFO['bfs_summary_rows']:,} validated BFS source runs across 12 datasets.",
    f'- **Unfinished configurations:** {missing_text}. These six cases are excluded; no values are estimated or imputed.',
    f"- **Zero-work measurements:** {MATRIX_INFO['zero_work_runs']:,}/{MATRIX_INFO['bfs_summary_rows']:,} recorded runs ({MATRIX_INFO['zero_work_fraction']:.1%}) traversed zero edges. The fixed-seed source sample is not degree-stratified.",
    '- **R-MAT interpretation:** RMAT_s17 through RMAT_s24 have zero median TEPS when all 16 sampled sources are included. `summary_aggregated.csv` also contains active-only median TEPS/elapsed time and the active-source count. Those active-only figures are based on small samples and are exploratory; they do not replace the all-source median.',
    f"- **Execution environment:** Notebook 03 recorded {logical_cpus} logical CPUs on one Colab VM. Runs with more MPI ranks than available logical CPUs are oversubscribed; do not interpret them as dedicated-core or multi-node scaling results.",
    '- **Per-level plot coverage:** each comparison uses the highest rank count common to all four ordering/communication combinations. For RMAT_s24, the common comparison is P=4 because higher-rank configurations are incomplete.',
    '', '### Fixed-seed source sample outcomes', '',
    '| Dataset | Sampled sources | Zero-work sources | Active sources | Zero-work share |',
    '|---|---:|---:|---:|---:|',
]
for row in sampling_df.sort_values('dataset').itertuples(index=False):
    coverage_md.append(f'| {row.dataset} | {row.sampled_sources} | {row.zero_work_sources} | {row.active_sources} | {row.zero_work_share:.1%} |')
coverage_md += [
    '',
    '*The main median-TEPS table includes zero-work measurements. `source_sampling_audit.csv` counts each fixed-seed source once per dataset; `summary_aggregated.csv` reports per-configuration zero-work counts and active-only derived statistics.*',
]
with md_path.open('a', encoding='utf-8') as handle:
    handle.write('\n'.join(coverage_md) + '\n')

# Build the DOCX, then replace its misleading "best configuration" table with a source-outcome audit.
docx_path = EXPORT_DIR / 'report' / 'Two_Page_Report.docx'
cb.run_stream([
    sys.executable, str(docx_script),
    '--input-dir', str(EXPORT_DIR / 'csv'),
    '--plots-dir', str(EXPORT_DIR / 'plots'),
    '--output', str(docx_path),
    '--environment', str(environment_file),
], cwd=WORK_DIR, timeout=900, check=True)

from docx import Document
from docx.shared import Pt

doc = Document(str(docx_path))
for para in doc.paragraphs:
    if para.text.strip() == 'Best measured configuration per dataset':
        para.text = 'Sampled source outcomes by dataset'
    elif para.text.startswith('The run bundle contains '):
        para.text = (
            f"The report includes {MATRIX_INFO['bfs_summary_rows']:,} validated BFS source runs from "
            f"{MATRIX_INFO['configurations']}/{MATRIX_INFO['planned_configurations']} planned configurations "
            f"({MATRIX_INFO['completion_fraction']:.1%}). Six RMAT_s24 configurations were not completed "
            "after an MPI process was killed (exit code 137); they are excluded and no values are estimated."
        )

if not doc.tables:
    raise RuntimeError('DOCX report is missing its source-summary table.')
table = doc.tables[0]
headers = ['Dataset', 'Runs', 'Zero-edge', 'Zero share', 'Median TEPS*']
for cell, value in zip(table.rows[0].cells, headers):
    cell.text = value
for i, row in enumerate(sampling_df.sort_values('dataset').itertuples(index=False), start=1):
    if i >= len(table.rows):
        table.add_row()
    g = summary_df[summary_df['dataset'] == row.dataset]
    values = [
        row.dataset,
        f'{len(g):,}',
        f'{int((g["traversed_edges"] == 0).sum()):,}',
        f'{(g["traversed_edges"] == 0).mean():.1%}',
        f'{g["teps"].median():,.0f}',
    ]
    for cell, value in zip(table.rows[i].cells, values):
        cell.text = value
while len(table.rows) > len(sampling_df) + 1:
    table._tbl.remove(table.rows[-1]._tr)
table.style = 'Light Shading Accent 1'

# Compact interpretation note, appended to the second page content.
doc.add_paragraph(
    f"Source-sampling/platform note: {MATRIX_INFO['zero_work_runs']:,} of {MATRIX_INFO['bfs_summary_rows']:,} "
    f"recorded runs ({MATRIX_INFO['zero_work_fraction']:.1%}) traversed zero edges. RMAT_s17–s24 therefore "
    "have zero all-source median TEPS; active-only statistics are in the companion aggregate CSV and have small "
    f"sample sizes. The runtime reported {logical_cpus} logical CPUs; P=4/8/16 were oversubscribed on a single VM. "
    "*Median TEPS includes zero-work runs."
)
doc.save(docx_path)

# Build the presentation; the temporary generator now prefers roadNet-CA plots and adds the coverage slide.
pptx_path = EXPORT_DIR / 'presentation' / 'Distributed_BFS_Results_Presentation.pptx'
cb.run_stream([
    sys.executable, str(presentation_script),
    '--results-dir', str(EXPORT_DIR),
    '--output', str(pptx_path),
], cwd=WORK_DIR, timeout=900, check=True)

# Independent final artifact checks.
expected_datasets = MATRIX_INFO['datasets']
for dataset in expected_datasets:
    for pattern in (
        'teps_*.png', 'per_level_*.png', 'collective_request_*.png',
        'cut_imbalance_*.png', 'remote_pairs_by_rank_*.png'
    ):
        matches = [p for p in (EXPORT_DIR / 'plots').glob(pattern) if dataset in p.stem]
        if not matches or any(p.stat().st_size == 0 for p in matches):
            raise RuntimeError(f'Required plot missing/empty for {dataset}: {pattern}')

required = [
    EXPORT_DIR / 'plots' / 'summary_aggregated.csv',
    EXPORT_DIR / 'plots' / 'per_level_derived.csv',
    EXPORT_DIR / 'csv' / 'source_sampling_audit.csv',
    EXPORT_DIR / 'csv' / 'configuration_coverage.csv',
    EXPORT_DIR / 'report' / 'Measured_Analysis_Report.md',
    docx_path,
    pptx_path,
]
for path in required:
    if not path.is_file() or path.stat().st_size == 0:
        raise RuntimeError(f'Required report output missing or empty: {path}')

from pptx import Presentation
checked_doc = Document(str(docx_path))
checked_deck = Presentation(str(pptx_path))
if len(checked_doc.paragraphs) < 5 or len(checked_doc.tables) < 1 or len(checked_doc.tables[0].rows) != 13:
    raise RuntimeError('DOCX report did not pass structural validation.')
if len(checked_deck.slides) < 15:
    raise RuntimeError(f'Presentation appears incomplete: only {len(checked_deck.slides)} slides.')
all_slide_text = '\n'.join(
    shape.text for slide in checked_deck.slides for shape in slide.shapes if hasattr(shape, 'text')
)
if '234 of 240' not in all_slide_text or '2 logical CPUs' not in all_slide_text:
    raise RuntimeError('Presentation is missing its partial-matrix/runtime disclosure.')
markdown_text = md_path.read_text(encoding='utf-8')
if '234 of 240' not in markdown_text and '234/240' not in markdown_text:
    raise RuntimeError('Markdown report is missing its partial-matrix disclosure.')

completion = {
    'input_run': 'full_matrix',
    'input_matrix_status': 'PARTIAL_VALIDATED',
    'input_configurations_validated': MATRIX_INFO['configurations'],
    'input_configurations_planned': MATRIX_INFO['planned_configurations'],
    'input_bfs_rows': MATRIX_INFO['bfs_summary_rows'],
    'input_missing_configurations': MATRIX_INFO['missing_configurations'],
    'input_zero_work_runs': MATRIX_INFO['zero_work_runs'],
    'input_zero_work_fraction': MATRIX_INFO['zero_work_fraction'],
    'datasets': expected_datasets,
    'charts_verified': sum(1 for p in (EXPORT_DIR / 'plots').glob('*.png') if p.stat().st_size > 0),
    'docx_paragraphs': len(checked_doc.paragraphs),
    'docx_table_rows': len(checked_doc.tables[0].rows),
    'pptx_slides': len(checked_deck.slides),
    'report_generation_success': True,
    'benchmark_matrix_complete': False,
}
cb.write_success_manifest(EXPORT_DIR, '04', {
    'source_fingerprint': source_hash,
    'completion': completion,
})
cb.create_success_zip(EXPORT_DIR, WORKSPACE, ARCHIVE_NAME)
print('\nREPORT REBUILD COMPLETE — PARTIAL INPUT MATRIX DISCLOSED.')
print(json.dumps(completion, indent=2))
print('The benchmark matrix remains partial; this SUCCESS.json records report-generation success only.')


$ /usr/bin/python3 -m pip install -q -r /content/Distributed_BFS_MPI/requirements.txt

$ /usr/bin/python3 /content/Distributed_BFS_MPI/scripts/analyze_results.py --input-dir /content/drive/MyDrive/Distributed_BFS_MPI_Colab/exports/reports_full_matrix_partial/csv --output-dir /content/drive/MyDrive/Distributed_BFS_MPI_Colab/exports/reports_full_matrix_partial/plots
Analyzed 3744 validated BFS runs across 12 datasets. Aggregates and plots: /content/drive/MyDrive/Distributed_BFS_MPI_Colab/exports/reports_full_matrix_partial/plots

$ /usr/bin/python3 /content/Distributed_BFS_MPI/scripts/generate_report.py --input-dir /content/drive/MyDrive/Distributed_BFS_MPI_Colab/exports/reports_full_matrix_partial/csv --output /content/drive/MyDrive/Distributed_BFS_MPI_Colab/exports/reports_full_matrix_partial/report/Measured_Analysis_Report.md --environment /content/drive/MyDrive/Distributed_BFS_MPI_Colab/exports/reports_full_matrix_partial/metadata/input_environment_summary.txt
/content/drive/MyDrive

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


REPORT REBUILD COMPLETE — PARTIAL INPUT MATRIX DISCLOSED.
{
  "input_run": "full_matrix",
  "input_matrix_status": "PARTIAL_VALIDATED",
  "input_configurations_validated": 234,
  "input_configurations_planned": 240,
  "input_bfs_rows": 3744,
  "input_missing_configurations": [
    "RMAT_s24_p16_original_blocking",
    "RMAT_s24_p16_original_nonblocking",
    "RMAT_s24_p16_random_blocking",
    "RMAT_s24_p16_random_nonblocking",
    "RMAT_s24_p8_random_blocking",
    "RMAT_s24_p8_random_nonblocking"
  ],
  "input_zero_work_runs": 2102,
  "input_zero_work_fraction": 0.5614316239316239,
  "datasets": [
    "RMAT_s16",
    "RMAT_s17",
    "RMAT_s18",
    "RMAT_s19",
    "RMAT_s20",
    "RMAT_s21",
    "RMAT_s22",
    "RMAT_s23",
    "RMAT_s24",
    "roadNet-CA",
    "soc-Epinions1",
    "web-Google"
  ],
  "charts_verified": 60,
  "docx_paragraphs": 20,
  "docx_table_rows": 13,
  "pptx_slides": 18,
  "report_generation_success": true,
  "benchmark_matrix_complete": false
}
The benchmark m

## Output

On success, the notebook downloads `04_full_matrix_partial_reports_results.zip` and saves a copy under `MyDrive/Distributed_BFS_MPI_Colab/downloads/`.

The archive contains the DOCX, PPTX, Markdown report, plots, reconstructed validated CSVs, source-sampling audit, configuration-coverage audit, and metadata documenting that the input benchmark matrix is partial. The original `runs/full_matrix/` folder is not modified.

**Presentation guidance:** use the report's caveats. The 234 completed configurations are validated, but the experiment is not a complete 240-configuration matrix. In particular, do not describe the 8/16-rank measurements as dedicated-core or multi-node scaling, and do not present the large-RMAT all-source median TEPS as meaningful throughput because most selected sources had zero traversed edges.